In [59]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.ensemble import RandomForestClassifier


In [38]:
data = pd.read_csv('bank-additional-full.csv', sep =';')
data.head(5)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

In [6]:
data['y'].value_counts(normalize=True)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [39]:
#. Cek kolom kategorikal mana saja yang punya
categorical_col = data.select_dtypes(include='str').columns.tolist()
for cols in categorical_col:
    unknown_count = (data[cols] =='unknown').sum()
    if unknown_count > 0:
        print(f"{cols}: {unknown_count} ({unknown_count/len(data)*100:.2f}%)")

job: 330 (0.80%)
marital: 80 (0.19%)
education: 1731 (4.20%)
default: 8597 (20.87%)
housing: 990 (2.40%)
loan: 990 (2.40%)


In [33]:
((data['housing'] == 'unknown') & (data['loan'] == 'unknown')).sum()

990

In [14]:
data.groupby('default')['y'].value_counts(normalize=True)

default  y  
no       no     0.87121
         yes    0.12879
unknown  no     0.94847
         yes    0.05153
yes      no     1.00000
Name: proportion, dtype: float64

In [40]:
data = data.drop(columns=['duration'])
data['was_contacted_before'] = (data['pdays'] != 999).astype(int)
data = data.drop(columns=['pdays'])

In [41]:
data['loan_info_missing'] = (data['housing'] == 'unknown').astype(int)

In [42]:
data['loan_info_missing'].sum()

990

In [43]:
numeric_cols = [
    'age', 'campaign', 'previous',
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed',
    'was_contacted_before', 'loan_info_missing'
]

categorical_nominal_cols = [
    'job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome'
]

categorical_ordinal_cols = ['education']

education_order = [
    'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
    'high.school', 'professional.course', 'university.degree', 'unknown'
]

In [47]:
numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])
nominal_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
ordinal_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(categories=[education_order]))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('nom', nominal_pipeline, categorical_nominal_cols),
    ('ord', ordinal_pipeline, categorical_ordinal_cols)
])

In [48]:
X = data.drop(columns=['y'])
y = data['y'].map({'no': 0, 'yes': 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [49]:
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clasifier', LogisticRegression(max_iter=1000, random_state=42))
    
])
full_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('clasifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age','job','marital',...,'nr.employed','was_contacted_before', 'loan_info_missing']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [50]:
y_pred = full_pipeline.predict(X_test)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.91      0.99      0.95      7310
           1       0.69      0.22      0.33       928

    accuracy                           0.90      8238
   macro avg       0.80      0.60      0.64      8238
weighted avg       0.88      0.90      0.88      8238

PR-AUC: 0.46578007867304494


In [56]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
# Cek beberapa titik supaya dapat gambaran trade-off-nya
for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
    idx = np.argmin(np.abs(thresholds - t))
    print(f"threshold={t:.1f} | precision={precisions[idx]:.3f} | recall={recalls[idx]:.3f}")

threshold=0.1 | precision=0.327 | recall=0.678
threshold=0.2 | precision=0.453 | recall=0.595
threshold=0.3 | precision=0.511 | recall=0.442
threshold=0.4 | precision=0.589 | recall=0.317
threshold=0.5 | precision=0.689 | recall=0.220


In [57]:
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)  # +1e-10 hindari pembagian 0
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Threshold terbaik (F1): {best_threshold:.3f}")
print(f"Precision: {precisions[best_idx]:.3f}, Recall: {recalls[best_idx]:.3f}, F1: {f1_scores[best_idx]:.3f}")

Threshold terbaik (F1): 0.203
Precision: 0.458, Recall: 0.593, F1: 0.516


In [58]:
y_pred_new = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_new))

              precision    recall  f1-score   support

           0       0.95      0.91      0.93      7310
           1       0.46      0.59      0.52       928

    accuracy                           0.87      8238
   macro avg       0.70      0.75      0.72      8238
weighted avg       0.89      0.87      0.88      8238



In [60]:
full_pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,       # coba isi sendiri, misal 100-300
        class_weight='balanced',
        random_state=42
    ))
])

full_pipeline_rf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age','job','marital',...,'nr.employed','was_contacted_before', 'loan_info_missing']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

In [61]:
y_pred_rf = full_pipeline_rf.predict(X_test)
y_proba_rf = full_pipeline_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("PR-AUC:", average_precision_score(y_test, y_proba_rf))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      7310
           1       0.46      0.49      0.47       928

    accuracy                           0.88      8238
   macro avg       0.70      0.71      0.70      8238
weighted avg       0.88      0.88      0.88      8238

PR-AUC: 0.39169811359305867


In [62]:
full_pipeline_rf2 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=10,           # batasi kedalaman
        min_samples_leaf=20,    # cegah leaf terlalu spesifik
        class_weight='balanced',
        random_state=42
    ))
])

full_pipeline_rf2.fit(X_train, y_train)

y_proba_rf2 = full_pipeline_rf2.predict_proba(X_test)[:, 1]
print("PR-AUC (RF tuned):", average_precision_score(y_test, y_proba_rf2))

PR-AUC (RF tuned): 0.49121726976926755


In [63]:
precisions_rf, recalls_rf, thresholds_rf = precision_recall_curve(y_test, y_proba_rf2)

f1_scores_rf = 2 * (precisions_rf * recalls_rf) / (precisions_rf + recalls_rf + 1e-10)
best_idx_rf = np.argmax(f1_scores_rf)
best_threshold_rf = thresholds_rf[best_idx_rf]

print(f"Threshold terbaik (F1): {best_threshold_rf:.3f}")
print(f"Precision: {precisions_rf[best_idx_rf]:.3f}, Recall: {recalls_rf[best_idx_rf]:.3f}, F1: {f1_scores_rf[best_idx_rf]:.3f}")

Threshold terbaik (F1): 0.696
Precision: 0.485, Recall: 0.581, F1: 0.528


In [64]:
y_pred_rf_new = (y_proba_rf2 >= best_threshold_rf).astype(int)
print(classification_report(y_test, y_pred_rf_new))

              precision    recall  f1-score   support

           0       0.95      0.92      0.93      7310
           1       0.48      0.58      0.53       928

    accuracy                           0.88      8238
   macro avg       0.72      0.75      0.73      8238
weighted avg       0.89      0.88      0.89      8238

